# 02-3. 積分 — 動かして確かめる

📖 解説: [`../03_integrals.md`](../03_integrals.md)

## このノートで触るもの
1. リーマン和を可視化 (短冊が面積になる)
2. SymPy で記号積分 / SciPy で数値積分
3. 二重積分
4. 期待値 = 積分 (正規分布)
5. JAX 版台形則

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (03_integrals.md)](../03_integrals.md)

In [ ]:
import numpy as np
import sympy as sp
import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt
from scipy import integrate
from ipywidgets import interact, IntSlider

%matplotlib inline

## 1. リーマン和 — 短冊で面積を近似

In [ ]:
def plot_riemann(n: int) -> None:
    """f(x) = x² の [0, 2] における面積を n 個の短冊で近似."""
    f = lambda x: x**2
    a, b = 0.0, 2.0
    xs_smooth = np.linspace(a, b, 200)
    
    # 短冊
    xs = np.linspace(a, b, n + 1)
    widths = (b - a) / n
    heights = f(xs[:-1])
    riemann_sum = np.sum(heights * widths)
    
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(xs_smooth, f(xs_smooth), 'b-', linewidth=2, label='f(x) = x²')
    ax.bar(xs[:-1], heights, width=widths, align='edge',
           color='orange', alpha=0.4, edgecolor='red', label=f'{n} 本の短冊')
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_title(f'n = {n},  リーマン和 = {riemann_sum:.6f}  (真の値 8/3 = {8/3:.6f})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

interact(plot_riemann, n=IntSlider(min=2, max=200, step=2, value=10));

## 2. SymPy で記号積分

In [ ]:
x = sp.Symbol('x')

# 不定積分
print('∫ x² dx     =', sp.integrate(x**2, x))
print('∫ sin(x) dx =', sp.integrate(sp.sin(x), x))
print('∫ e^x dx    =', sp.integrate(sp.exp(x), x))
print('∫ 1/x dx    =', sp.integrate(1/x, x))

# 定積分
print()
print('∫₀² x² dx       =', sp.integrate(x**2, (x, 0, 2)))
print('∫₀^π sin(x) dx  =', sp.integrate(sp.sin(x), (x, 0, sp.pi)))
print('∫₀^∞ e^(-x) dx  =', sp.integrate(sp.exp(-x), (x, 0, sp.oo)))

## 3. SciPy で数値積分

In [ ]:
# シンプルな例
result, error = integrate.quad(lambda x: x**2, 0, 2)
print(f'∫₀² x² dx = {result:.10f}  (誤差推定 {error:.2e})')

# SymPy で解けない関数
result, _ = integrate.quad(lambda x: np.exp(-x**2), -10, 10)
print(f'∫₋₁₀^₁₀ e^(-x²) dx = {result:.10f}  (理論 √π = {np.sqrt(np.pi):.10f})')

# 無限区間も OK
result, _ = integrate.quad(lambda x: 1 / (1 + x**2), -np.inf, np.inf)
print(f'∫₋∞^∞ 1/(1+x²) dx = {result:.10f}  (理論 π = {np.pi:.10f})')

## 4. 二重積分

In [ ]:
# ∫₀¹ ∫₀¹ xy dxdy = 1/4
result, _ = integrate.dblquad(lambda x, y: x * y, 0, 1, 0, 1)
print(f'∫₀¹ ∫₀¹ xy dxdy = {result}  (理論 0.25)')

# ガウス分布の規格化 (2次元)
def gaussian_2d(x, y):
    return np.exp(-(x**2 + y**2) / 2) / (2 * np.pi)

result, _ = integrate.dblquad(gaussian_2d, -10, 10, -10, 10)
print(f'2D ガウス分布の積分 = {result:.6f}  (理論 1.0)')

## 5. 期待値 = 積分

In [ ]:
# 標準正規分布 N(0, 1) の期待値
def integrand(x):
    return x * np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)

ev, _ = integrate.quad(integrand, -np.inf, np.inf)
print(f'E[X] for N(0,1) = {ev:.6f}  (理論 0)')

# 分散も積分で
def integrand_var(x):
    return x**2 * np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)

var, _ = integrate.quad(integrand_var, -np.inf, np.inf)
print(f'Var[X] for N(0,1) = {var:.6f}  (理論 1)')

## 6. JAX で台形則 (勾配が必要な場面用)

In [ ]:
@jax.jit
def trapezoid(f, a, b, n=1000):
    """台形則."""
    xs = jnp.linspace(a, b, n + 1)
    ys = jax.vmap(f)(xs)
    h = (b - a) / n
    return h * (jnp.sum(ys) - 0.5 * (ys[0] + ys[-1]))

result = trapezoid(lambda x: x**2, 0.0, 2.0)
print(f'JAX 台形則: ∫₀² x² dx = {float(result):.6f}  (理論 8/3 = {8/3:.6f})')

# 積分の境界の勾配 (上端 b について微分)
@jax.jit
def integral_upper(b):
    return trapezoid(lambda x: x**2, 0.0, b, n=10000)

df = jax.grad(integral_upper)
print(f'd/db ∫₀^b x² dx at b=2: {float(df(2.0)):.4f}  (理論 b² = 4)')
print('→ 微積分学の基本定理: ∫₀^b x² dx の b 微分 = b²')

## まとめ
- 短冊を細くすると本物の面積に近づく (リーマン和の極限 = 積分)
- SymPy で記号積分、SciPy で数値積分が現実的
- 期待値・分散はすべて積分で定義される
- JAX なら積分自体の勾配が取れる (物理 + ML 統合の鍵)

## 次へ
→ [`04_multivariable.ipynb`](04_multivariable.ipynb)  解説 → [`../04_multivariable.md`](../04_multivariable.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`02_derivatives.ipynb`](02_derivatives.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`04_multivariable.ipynb`](04_multivariable.ipynb) |